# Credit card balances: EDA and feature engineering

This notebook studies `credit_card_balance.csv`, a month-by-month record of each applicant's earlier Home Credit credit cards, and turns it into features, one row per applicant. It is the last of the history tables.

## Where things stand

| Table | Notebooks | Result |
|---|---|---|
| Credit bureau | 03, 04 | Adopted. Cross-validated ROC AUC 0.7685 to 0.7779 |
| Earlier applications | 05, 06 | Adopted. 0.7779 to 0.7856 |
| Instalment payments | 07, 08 | Adopted. 0.7856 to 0.7932; Kaggle private 0.7934 |
| Monthly balances of cash and point-of-sale loans | 09, 10 | Left out: +0.00007, no better than scrambled columns |

## Why this table might be different from the last one

Notebook 10's table failed because it described the same loans as tables already in the model. This one describes **cards**, which the model sees only roughly:

- `previous_application` knows a card was applied for and approved, and its features count approved revolving loans.
- `installments_payments` records card payments, but notebook 07 found that on a card the "amount due" looks like a record of what was paid. It could count card instalments and lateness on them, and little else.
- What neither has is how the card was **used**: the balance against the limit, cash drawn at ATMs, whether the minimum payment was met. Utilisation is one of the best-known signals in consumer credit.

So there is a reason to expect something new here, and also a reason for caution. About one applicant in five has a card with payments on record, so whatever this table adds, it adds for a minority.

## Inputs

| File | Rows | One row is | Key |
|---|---:|---|---|
| `data/raw/credit_card_balance.csv` | about 3.8 million | One month of one card | `SK_ID_PREV` and `MONTHS_BALANCE` |
| `data/raw/previous_application.csv` | about 1.7 million | One earlier application | `SK_ID_PREV` |
| `data/processed/*_features.parquet` | | The saved features of the earlier tables | `SK_ID_CURR` |

## What this notebook does

The nine stages of notebooks 03, 05, 07 and 09: structure and keys, coverage and target, column audit, time audit, signal and whether it is new, aggregation design, candidate features with an independent rebuild, screening, and save.

## Rules followed throughout

Unchanged: the holdout stays closed and only development outcomes are looked at; a feature uses only the applicant's own rows; nothing is learned from the target; only information from before the application is used; every column is audited for whether it is believable and whether it is supplied on the same terms in train and test; rare events are judged by what they do for the applicants they describe.

## Out of scope

Whether the features improve the model, which is notebook 12.

## Stage 1: Structure and keys

No target is used in this stage.

In [1]:
import gc, sys, time, json
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 240)
SEED = 42
RAW, PROCESSED = '../data/raw', '../data/processed'
LOAN, MONTH = 'SK_ID_PREV', 'MONTHS_BALANCE'

started = time.time()
cc = pd.read_csv(f'{RAW}/credit_card_balance.csv')
print(f"credit_card_balance.csv: {cc.shape[0]:,} rows, {cc.shape[1]} columns, {cc.memory_usage(deep=True).sum() / 1e6:,.0f} MB, read in {time.time() - started:.1f} seconds")

overview = pd.DataFrame({'dtype': cc.dtypes.astype(str), 'missing_pct': cc.isna().mean() * 100, 'distinct': cc.nunique(), 'min': cc.min(numeric_only=True),
                         'median': cc.median(numeric_only=True), 'max': cc.max(numeric_only=True), 'negative_pct': (cc.select_dtypes('number') < 0).mean() * 100})
print(overview.to_string(float_format=lambda v: f'{v:,.2f}'))
print('\nstatus:', cc['NAME_CONTRACT_STATUS'].value_counts().to_dict())

credit_card_balance.csv: 3,840,312 rows, 23 columns, 730 MB, read in 14.3 seconds


                              dtype  missing_pct  distinct          min       median          max  negative_pct
AMT_BALANCE                 float64         0.00   1347904  -420,250.18         0.00 1,505,902.19          0.06
AMT_CREDIT_LIMIT_ACTUAL       int64         0.00       181         0.00   112,500.00 1,350,000.00          0.00
AMT_DRAWINGS_ATM_CURRENT    float64        19.52      2267    -6,827.31         0.00 2,115,000.00          0.00
AMT_DRAWINGS_CURRENT        float64         0.00    187005    -6,211.62         0.00 2,287,098.31          0.00
AMT_DRAWINGS_OTHER_CURRENT  float64        19.52      1832         0.00         0.00 1,529,847.00          0.00
AMT_DRAWINGS_POS_CURRENT    float64        19.52    168748         0.00         0.00 2,239,274.16          0.00
AMT_INST_MIN_REGULARITY     float64         7.95    312266         0.00         0.00   202,882.01          0.00
AMT_PAYMENT_CURRENT         float64        20.00    163209         0.00     2,702.70 4,289,207.45       

### 1.1 What is one row, and how the table links

In [2]:
key_checks = {
    'no missing key or month': bool(cc[[LOAN, 'SK_ID_CURR', MONTH]].notna().all().all()),
    'no fully duplicated row': int(cc.duplicated().sum()) == 0,
    'every card belongs to one applicant': int(cc.groupby(LOAN)['SK_ID_CURR'].nunique().max()) == 1,
    'card and month identify a row': not cc.duplicated([LOAN, MONTH]).any(),
    'no month on or after the application': bool((cc[MONTH] < 0).all()),
}
for name, ok in key_checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {name}")
print(f"months from {cc[MONTH].min()} to {cc[MONTH].max()}")
cards = cc.groupby(LOAN).agg(applicant=('SK_ID_CURR', 'first'), months=(MONTH, 'size'), first_month=(MONTH, 'min'), last_month=(MONTH, 'max'))
cards['gaps'] = cards['last_month'] - cards['first_month'] + 1 - cards['months']
print(f"cards: {len(cards):,}; months per card: median {cards['months'].median():.0f}, 99th percentile {cards['months'].quantile(0.99):.0f}; cards without a gap in their months: {(cards['gaps'] == 0).mean():.2%}")
per_applicant = cards.groupby('applicant').size()
print(f"cards per applicant with any: {per_applicant.value_counts().head(5).to_dict()}")

split = pd.read_parquet(f'{PROCESSED}/application_split.parquet')
TEST_IDS = pd.read_parquet(f'{PROCESSED}/application_test_features.parquet', columns=[]).index
DEV_IDS, HOLDOUT_IDS, TRAIN_IDS = split.index[split['split'] == 'dev'], split.index[split['split'] == 'holdout'], split.index
PARTS = {'train: development': DEV_IDS, 'train: holdout': HOLDOUT_IDS, 'test': TEST_IDS}
part_of = pd.Series('neither file', index=pd.Index(cc['SK_ID_CURR'].unique()))
for label, ids in PARTS.items():
    part_of[part_of.index.isin(ids)] = label
cc_part = cc['SK_ID_CURR'].map(part_of)
coverage = pd.DataFrame({label: {'applicants': len(ids), 'with_a_card_pct': ids.isin(per_applicant.index).mean() * 100, 'rows': int((cc_part == label).sum())} for label, ids in PARTS.items()}).T
print('\n' + coverage.to_string(float_format=lambda v: f'{v:,.2f}'))
print(f"applicants here in neither application file: {int((part_of == 'neither file').sum())}")

sys.path.insert(0, '../src/features')
from previous_application import clean_previous
prev = clean_previous(pd.read_csv(f'{RAW}/previous_application.csv'))
prev_ids = prev.set_index('SK_ID_PREV')
cards['in_prev'] = cards.index.isin(prev_ids.index)
cards['part'] = cards['applicant'].map(part_of)
print(f"\ncards found in previous_application: {cards['in_prev'].mean():.2%}; by part {(cards.groupby('part')['in_prev'].mean() * 100).round(2).to_dict()}")
print('type and status of their applications:', prev_ids.loc[cards.index[cards['in_prev']], 'NAME_CONTRACT_TYPE'].value_counts().to_dict(), prev_ids.loc[cards.index[cards['in_prev']], 'NAME_CONTRACT_STATUS'].value_counts().to_dict())

# the supply test: of each applicant's approved revolving loans, the share with monthly rows here
approved = prev[(prev['NAME_CONTRACT_STATUS'] == 'Approved') & (prev['NAME_CONTRACT_TYPE'] == 'Revolving loans')].copy()
approved['has_rows'] = approved['SK_ID_PREV'].isin(cards.index)
known = approved[approved['SK_ID_CURR'].isin(TRAIN_IDS) | approved['SK_ID_CURR'].isin(TEST_IDS)]
share = known.groupby('SK_ID_CURR')['has_rows'].mean()
is_test = share.index.isin(TEST_IDS).astype(int)
a = roc_auc_score(is_test, share.values)
print(f"\napproved revolving loans of train and test applicants: {len(known):,}; with monthly rows here: {known['has_rows'].mean():.2%}")
print(f"supply test: share with rows, train {share[is_test == 0].mean():.3f}, test {share[is_test == 1].mean():.3f}; AUC {max(a, 1 - a):.3f}   (bureau_balance 0.82; POS_CASH_balance 0.503)")
age = pd.cut(-known['DAYS_DECISION'] / 365.25, [0, 1, 2, 4, 6, 8, 100], labels=['under 1 year', '1 to 2', '2 to 4', '4 to 6', '6 to 8', 'over 8'])
kpart = np.where(known['SK_ID_CURR'].isin(TEST_IDS), 'test', 'train')
print('share of approved revolving loans with rows, by age of the decision (%):')
print(pd.crosstab(age, kpart, values=known['has_rows'], aggfunc='mean').mul(100).round(2).to_string())

# against the payment table: card loans there (calendar version 0) and here
inst_cards = pd.read_csv(f'{RAW}/installments_payments.csv', usecols=['SK_ID_PREV', 'SK_ID_CURR', 'NUM_INSTALMENT_VERSION'])
inst_cards = inst_cards[inst_cards['NUM_INSTALMENT_VERSION'] == 0].groupby('SK_ID_PREV')['SK_ID_CURR'].first()
print(f"\ncard loans with payments on record: {len(inst_cards):,}; of these with monthly rows here: {inst_cards.index.isin(cards.index).mean():.2%}; cards here with payments there: {cards.index.isin(inst_cards.index).mean():.2%}")
has_card = pd.Series(TRAIN_IDS.append(TEST_IDS).isin(per_applicant.index), index=TRAIN_IDS.append(TEST_IDS))
has_card_payments = pd.Series(has_card.index.isin(inst_cards.unique()), index=has_card.index)
print('applicants (train and test) by card rows here and card payments there (% of all):')
print((pd.crosstab(has_card.map({True: 'card rows here', False: 'none here'}), has_card_payments.map({True: 'card payments there', False: 'none there'}), normalize='all') * 100).round(2).to_string())
del inst_cards
gc.collect()

PASS  no missing key or month
PASS  no fully duplicated row
PASS  every card belongs to one applicant
PASS  card and month identify a row
PASS  no month on or after the application
months from -96 to -1


cards: 104,307; months per card: median 21, 99th percentile 96; cards without a gap in their months: 99.99%
cards per applicant with any: {1: 102818, 2: 732, 3: 7, 4: 1}



                    applicants  with_a_card_pct         rows
train: development  246,008.00            28.33 2,585,374.00
train: holdout       61,503.00            28.00   642,591.00
test                 48,744.00            34.16   612,347.00
applicants here in neither application file: 0



cards found in previous_application: 89.10%; by part {'test': 86.92, 'train: development': 89.49, 'train: holdout': 89.64}
type and status of their applications: {'Revolving loans': 92935} {'Approved': 92935}



approved revolving loans of train and test applicants: 97,738; with monthly rows here: 95.09%
supply test: share with rows, train 0.958, test 0.961; AUC 0.502   (bureau_balance 0.82; POS_CASH_balance 0.503)
share of approved revolving loans with rows, by age of the decision (%):


col_0           test  train
DAYS_DECISION              
under 1 year   93.70  92.16
1 to 2         94.48  94.07
2 to 4         96.65  97.37
4 to 6         94.18  93.87
6 to 8         99.82  99.98



card loans with payments on record: 72,678; of these with monthly rows here: 99.71%; cards here with payments there: 69.47%
applicants (train and test) by card rows here and card payments there (% of all):


col_0           card payments there  none there
row_0                                          
card rows here                20.26        8.81
none here                      0.05       70.88


20

### 1.2 The months before the application

In [3]:
last = pd.DataFrame({label: cc.loc[cc['SK_ID_CURR'].isin(ids), MONTH].value_counts().reindex(range(-24, 0)).fillna(0) for label, ids in [('train', TRAIN_IDS), ('test', TEST_IDS)]})
last = last / last.loc[-24:-13].mean()
print('rows per month, relative to months -24 to -13:')
print(last.loc[[-12, -6, -3, -2, -1]].round(3).T.to_string())
active2 = cc.loc[(cc[MONTH] == -2) & (cc['NAME_CONTRACT_STATUS'] == 'Active'), [LOAN, 'SK_ID_CURR']]
has1 = active2[LOAN].isin(cc.loc[cc[MONTH] == -1, LOAN])
for label, ids in [('train', TRAIN_IDS), ('test', TEST_IDS)]:
    m = active2['SK_ID_CURR'].isin(ids).values
    print(f"{label}: cards active in month -2 with a row in month -1: {has1[m].mean():.1%}")
active5 = cc.loc[(cc[MONTH] == -5) & (cc['NAME_CONTRACT_STATUS'] == 'Active'), LOAN]
print(f"cards active in month -5 with a row in month -4: {active5.isin(cc.loc[cc[MONTH] == -4, LOAN]).mean():.1%}")
print(f"cards whose last month is -1: {(cards['last_month'] == -1).mean():.1%}; median last month {cards['last_month'].median():.0f}")

rows per month, relative to months -24 to -13:
MONTHS_BALANCE    -12    -6     -3     -2     -1 
train           1.236  1.666  1.722  1.627  1.063
test            1.393  1.663  1.562  1.458  1.009
train: cards active in month -2 with a row in month -1: 66.7%
test: cards active in month -2 with a row in month -1: 71.5%


cards active in month -5 with a row in month -4: 100.0%
cards whose last month is -1: 59.8%; median last month -1


## Stage 2: Coverage and target

Development applicants only.

In [4]:
y_dev = pd.read_parquet(f'{PROCESSED}/application_train_features.parquet', columns=['TARGET'])['TARGET'].loc[DEV_IDS]
BASE_RATE = y_dev.mean()
def rate_table(groups):
    g = groups.reindex(DEV_IDS)
    t = pd.DataFrame({'applicants': y_dev.groupby(g, observed=True).size(), 'default_pct': y_dev.groupby(g, observed=True).mean() * 100})
    t['pct_of_applicants'] = t['applicants'] / len(y_dev) * 100
    t['standard_error'] = np.sqrt(t['default_pct'] * (100 - t['default_pct']) / t['applicants'])
    t['ratio_to_average'] = t['default_pct'] / (BASE_RATE * 100)
    return t
kind = pd.Series('no card on record', index=has_card.index)
kind[has_card.values & has_card_payments.values] = 'card rows and card payments'
kind[has_card.values & ~has_card_payments.values] = 'card rows only'
kind[~has_card.values & has_card_payments.values] = 'card payments only'
print(f"development applicants {len(y_dev):,}; default rate {BASE_RATE:.2%}\n")
print(rate_table(kind).to_string(float_format=lambda v: f'{v:,.2f}'))

# what the model already knows about card holders: does it predict their rate?
inst_dev = pd.read_parquet(f'{PROCESSED}/installments_train_features.parquet', columns=['INST_CARD_N_INSTALMENTS', 'INST_CARD_ACTIVE_90D']).loc[DEV_IDS]
print('\namong applicants with card rows here, by card instalments on record in the payment table (a feature the model has):')
with_rows = has_card.reindex(DEV_IDS)
print(rate_table(pd.cut(inst_dev['INST_CARD_N_INSTALMENTS'].where(with_rows), [-1, 0, 12, 36, np.inf], labels=['none', '1 to 12', '13 to 36', 'over 36'])).to_string(float_format=lambda v: f'{v:,.2f}'))

development applicants 246,008; default rate 8.07%



                             applicants  default_pct  pct_of_applicants  standard_error  ratio_to_average
card payments only                  131         9.16               0.05            2.52              1.13
card rows and card payments       48979        10.07              19.91            0.14              1.25
card rows only                    20704         5.37               8.42            0.16              0.66
no card on record                176194         7.83              71.62            0.06              0.97

among applicants with card rows here, by card instalments on record in the payment table (a feature the model has):


                         applicants  default_pct  pct_of_applicants  standard_error  ratio_to_average
INST_CARD_N_INSTALMENTS                                                                              
none                          20518         5.38               8.34            0.16              0.67
1 to 12                        5014        13.50               2.04            0.48              1.67
13 to 36                      16287        13.04               6.62            0.26              1.62
over 36                       27678         7.70              11.25            0.16              0.95


## Stage 3: Column-by-column audit

The money columns, the counts of drawings, the status, and days past due. No target is used.

In [5]:
amounts = ['AMT_BALANCE', 'AMT_CREDIT_LIMIT_ACTUAL', 'AMT_DRAWINGS_ATM_CURRENT', 'AMT_DRAWINGS_CURRENT', 'AMT_DRAWINGS_OTHER_CURRENT', 'AMT_DRAWINGS_POS_CURRENT', 'AMT_INST_MIN_REGULARITY',
           'AMT_PAYMENT_CURRENT', 'AMT_PAYMENT_TOTAL_CURRENT', 'AMT_RECEIVABLE_PRINCIPAL', 'AMT_RECIVABLE', 'AMT_TOTAL_RECEIVABLE']
counts = ['CNT_DRAWINGS_ATM_CURRENT', 'CNT_DRAWINGS_CURRENT', 'CNT_DRAWINGS_OTHER_CURRENT', 'CNT_DRAWINGS_POS_CURRENT', 'CNT_INSTALMENT_MATURE_CUM']
audit = pd.DataFrame({c: {'missing_pct': cc[c].isna().mean() * 100, 'zero_pct': (cc[c] == 0).mean() * 100, 'negative_pct': (cc[c] < 0).mean() * 100,
                          'p50': cc[c].median(), 'p99': cc[c].quantile(0.99), 'max': cc[c].max(),
                          'missing_train_pct': cc.loc[cc_part != 'test', c].isna().mean() * 100, 'missing_test_pct': cc.loc[cc_part == 'test', c].isna().mean() * 100} for c in amounts + counts}).T
print(audit.to_string(float_format=lambda v: f'{v:,.2f}'))

# which columns go missing together?
miss = cc[amounts + counts].isna()
print('\ncolumns missing on the same rows:', miss.T.drop_duplicates().index.tolist())
print(f"rows with the drawing columns missing: {miss['AMT_DRAWINGS_ATM_CURRENT'].mean():.2%}; of them with CNT_DRAWINGS_CURRENT above 0: {(cc.loc[miss['AMT_DRAWINGS_ATM_CURRENT'], 'CNT_DRAWINGS_CURRENT'] > 0).mean():.2%}")

consistency = {
    'drawings: ATM + POS + other equals the total (where all present, within 1)': ((cc['AMT_DRAWINGS_ATM_CURRENT'] + cc['AMT_DRAWINGS_POS_CURRENT'] + cc['AMT_DRAWINGS_OTHER_CURRENT'] - cc['AMT_DRAWINGS_CURRENT']).abs() <= 1)[cc[['AMT_DRAWINGS_ATM_CURRENT', 'AMT_DRAWINGS_POS_CURRENT', 'AMT_DRAWINGS_OTHER_CURRENT']].notna().all(axis=1)].mean(),
    'counts: ATM + POS + other equals the total count': ((cc['CNT_DRAWINGS_ATM_CURRENT'] + cc['CNT_DRAWINGS_POS_CURRENT'] + cc['CNT_DRAWINGS_OTHER_CURRENT']) == cc['CNT_DRAWINGS_CURRENT'])[cc[['CNT_DRAWINGS_ATM_CURRENT', 'CNT_DRAWINGS_POS_CURRENT', 'CNT_DRAWINGS_OTHER_CURRENT']].notna().all(axis=1)].mean(),
    'total receivable equals receivable (within 1)': ((cc['AMT_TOTAL_RECEIVABLE'] - cc['AMT_RECIVABLE']).abs() <= 1).mean(),
    'receivable equals balance (within 1)': ((cc['AMT_RECIVABLE'] - cc['AMT_BALANCE']).abs() <= 1).mean(),
    'payment total is at least the payment (where both present)': (cc['AMT_PAYMENT_TOTAL_CURRENT'] >= cc['AMT_PAYMENT_CURRENT'] - 1)[cc['AMT_PAYMENT_CURRENT'].notna()].mean(),
    'balance at or under the limit (where the limit is above 0)': (cc['AMT_BALANCE'] <= cc['AMT_CREDIT_LIMIT_ACTUAL'])[cc['AMT_CREDIT_LIMIT_ACTUAL'] > 0].mean(),
}
print()
for name, share_ok in consistency.items():
    print(f"{share_ok:8.2%}  {name}")
print(f"\nlimit of zero: {(cc['AMT_CREDIT_LIMIT_ACTUAL'] == 0).mean():.2%} of rows; balance above twice the limit: {(cc['AMT_BALANCE'] > 2 * cc['AMT_CREDIT_LIMIT_ACTUAL'].where(cc['AMT_CREDIT_LIMIT_ACTUAL'] > 0)).mean():.3%}")
util = cc['AMT_BALANCE'] / cc['AMT_CREDIT_LIMIT_ACTUAL'].where(cc['AMT_CREDIT_LIMIT_ACTUAL'] > 0)
print(f"balance over limit, quantiles: {util.quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.99]).round(3).to_dict()}")
print(f"negative balances: {(cc['AMT_BALANCE'] < 0).mean():.3%} of rows, median {cc.loc[cc['AMT_BALANCE'] < 0, 'AMT_BALANCE'].median():.1f}; negative drawings: {(cc['AMT_DRAWINGS_ATM_CURRENT'] < 0).sum() + (cc['AMT_DRAWINGS_CURRENT'] < 0).sum()} values")
active = cc['NAME_CONTRACT_STATUS'] == 'Active'
paid = cc['AMT_PAYMENT_TOTAL_CURRENT']
min_due = cc['AMT_INST_MIN_REGULARITY']
print(f"months with a minimum payment above 0: {(min_due > 0).mean():.1%}; of those, paid at least the minimum: {(paid >= min_due - 1)[min_due > 0].mean():.1%}; paid nothing: {(paid == 0)[min_due > 0].mean():.1%}")

dpd, dpd_def = cc['SK_DPD'], cc['SK_DPD_DEF']
print(f"\nmonths past due: {(dpd > 0).mean():.2%}; under the tolerance rule: {(dpd_def > 0).mean():.2%}; SK_DPD_DEF above SK_DPD: {int((dpd_def > dpd).sum())}")
print(f"past due but ignored by the tolerance rule: days past due median {dpd[(dpd > 0) & (dpd_def == 0)].median():.0f}; balance median {cc.loc[(dpd > 0) & (dpd_def == 0), 'AMT_BALANCE'].median():,.0f}")
print('status by part (% of rows):')
print((pd.crosstab(cc['NAME_CONTRACT_STATUS'], np.where(cc_part == 'test', 'test', 'train'), normalize='columns') * 100).round(3).to_string())

def supply(flag):
    share = pd.Series(np.asarray(flag, dtype=float)).groupby(cc['SK_ID_CURR'].values).mean()
    share = share[share.index.isin(TRAIN_IDS) | share.index.isin(TEST_IDS)]
    t = share.index.isin(TEST_IDS).astype(int)
    a = roc_auc_score(t, share.values)
    return {'train_pct': share[t == 0].mean() * 100, 'test_pct': share[t == 1].mean() * 100, 'auc': max(a, 1 - a)}
facts = {'drawing amounts missing': miss['AMT_DRAWINGS_ATM_CURRENT'], 'payment missing': cc['AMT_PAYMENT_CURRENT'].isna(), 'minimum payment missing': min_due.isna(),
         'balance above 0': cc['AMT_BALANCE'] > 0, 'limit of zero': cc['AMT_CREDIT_LIMIT_ACTUAL'] == 0, 'any ATM drawing': cc['AMT_DRAWINGS_ATM_CURRENT'] > 0,
         'paid under the minimum': (paid < min_due - 1) & (min_due > 0), 'past due': dpd > 0, 'past due, tolerance rule': dpd_def > 0, 'status Active': active,
         'status Completed': cc['NAME_CONTRACT_STATUS'] == 'Completed', 'balance over the limit': util > 1}
supply_table = pd.DataFrame({k: supply(v) for k, v in facts.items()}).T
print('\nsupply test, per fact:')
print(supply_table.sort_values('auc', ascending=False).to_string(float_format=lambda v: f'{v:,.3f}'))
print(f"facts at 0.55 or more: {list(supply_table.index[supply_table['auc'] >= 0.55])}")

                            missing_pct  zero_pct  negative_pct        p50        p99          max  missing_train_pct  missing_test_pct
AMT_BALANCE                        0.00     56.15          0.06       0.00 466,295.12 1,505,902.19               0.00              0.00
AMT_CREDIT_LIMIT_ACTUAL            0.00     19.63          0.00 112,500.00 765,000.00 1,350,000.00               0.00              0.00
AMT_DRAWINGS_ATM_CURRENT          19.52     69.41          0.00       0.00 135,000.00 2,115,000.00              18.77             23.53
AMT_DRAWINGS_CURRENT               0.00     83.94          0.00       0.00 157,500.00 2,287,098.31               0.00              0.00
AMT_DRAWINGS_OTHER_CURRENT        19.52     80.15          0.00       0.00       0.00 1,529,847.00              18.77             23.53
AMT_DRAWINGS_POS_CURRENT          19.52     73.58          0.00       0.00  77,080.61 2,239,274.16              18.77             23.53
AMT_INST_MIN_REGULARITY            7.95     50.2


columns missing on the same rows: ['AMT_BALANCE', 'AMT_DRAWINGS_ATM_CURRENT', 'AMT_INST_MIN_REGULARITY', 'AMT_PAYMENT_CURRENT']
rows with the drawing columns missing: 19.52%; of them with CNT_DRAWINGS_CURRENT above 0: 0.00%



  99.78%  drawings: ATM + POS + other equals the total (where all present, within 1)
 100.00%  counts: ATM + POS + other equals the total count
  99.71%  total receivable equals receivable (within 1)
  84.55%  receivable equals balance (within 1)
  61.73%  payment total is at least the payment (where both present)
  89.17%  balance at or under the limit (where the limit is above 0)

limit of zero: 19.63% of rows; balance above twice the limit: 0.008%


balance over limit, quantiles: {0.1: 0.0, 0.25: 0.0, 0.5: 0.011, 0.75: 0.869, 0.9: 1.005, 0.99: 1.069}
negative balances: 0.061% of rows, median -359.0; negative drawings: 4 values


months with a minimum payment above 0: 41.8%; of those, paid at least the minimum: 86.7%; paid nothing: 8.5%

months past due: 3.99%; under the tolerance rule: 2.33%; SK_DPD_DEF above SK_DPD: 0
past due but ignored by the tolerance rule: days past due median 397; balance median 210
status by part (% of rows):


col_0                   test   train
NAME_CONTRACT_STATUS                
Active                95.005  96.552
Approved               0.000   0.000
Completed              4.717   3.099
Demand                 0.085   0.026
Refused                0.000   0.000
Sent proposal          0.009   0.014
Signed                 0.183   0.308



supply test, per fact:
                          train_pct  test_pct   auc
minimum payment missing       5.141     3.791 0.554
any ATM drawing              14.356    11.276 0.542
balance over the limit        9.461     7.277 0.537
balance above 0              44.355    39.145 0.536
drawing amounts missing      32.894    37.749 0.526
payment missing              35.743    39.914 0.525
paid under the minimum        6.641     6.104 0.523
past due, tolerance rule      1.144     0.828 0.520
past due                      2.079     1.607 0.520
status Completed              2.918     5.073 0.519
status Active                96.691    94.612 0.514
limit of zero                12.046    12.497 0.508
facts at 0.55 or more: ['minimum payment missing']


### Stage 1 to 3 findings

**The table**
- 3,840,312 rows and 23 columns. One row is one month of one card: card and month identify a row, every card belongs to one applicant, and 99.99% of cards have every month between their first and last. No month is on or after the application, and there is no month 0.
- 104,307 cards, almost always one per applicant (98.6% of those with any). A card's record is a median 21 months long.

**Supplied on the same terms, with one difference in who has a card**
- Of approved revolving loans in `previous_application`, 95% have monthly rows here, 0.958 of an applicant's in train against 0.961 in test. The supply test gives **0.502**.
- But more test applicants have a card at all: **34.2% against 28.3%** in train. That is not how the data was supplied; it is a difference in the applicants, of the kind found for every Home Credit table. It means features from this table will be present more often in the test file.
- Twelve facts about months were tested. One is flagged: rows with no minimum payment recorded are 5.1% of an applicant's months in train and 3.8% in test (0.554). Three more are near the line: any ATM drawing (0.542), balance over the limit (0.537), balance above zero (0.536). Test card holders use their cards a little less. Stage 8 checks every feature for shift.
- The drawing columns are missing on 18.8% of rows in train and 23.5% in test. Where they are missing, no drawing was counted in the month, so a missing drawing amount is a month with no drawing.

**The month before the application is partly missing, as in notebook 09**
- Of cards active in month -2, 67% have a row in month -1 in train and 72% in test. Three months earlier the same check gives 100%. Features use each card's latest row within three months, not month -1.

**It describes cards the model knows only roughly, and some it does not see at all**
- 99.7% of the cards with payments in `installments_payments` have monthly rows here. But **30% of the cards here have no payments there**. Those belong to the 8.4% of development applicants who have a card and never paid anything on it.

| Applicants | Share | Default rate |
|---|---:|---:|
| A card here and card payments there | 19.9% | 10.07% |
| A card here, no card payments there | 8.4% | **5.37%** |
| No card on record | 71.6% | 7.83% |

- An unused card is a mark of a safe applicant. The model can already see this group, as applicants approved for a revolving loan with no card payments, and it does: among card holders, those with no card instalments on record default at 5.4%, the same rate.
- Among card holders who do pay, the model's card features already separate a short history (13% default) from a long one (7.7%).

**The columns**
- The limit is zero on 20% of rows: cards not yet issued or closed. The balance is above the limit on 11% of rows with a limit, and by up to 7% at the 99th percentile. Half of the months have a balance of zero.
- In 42% of months a minimum payment is due. In those, 86.7% are paid at least the minimum and 8.5% are paid nothing.
- 4.0% of months are past due and 2.3% under the tolerance rule. As in notebook 09, the months the tolerance rule ignores are small balances (median 210) counted for a long time (median 397 days).
- The drawing amounts add up to their total in 99.8% of rows and the drawing counts in 100%. The receivable columns duplicate the balance.

**Decisions**
- Arrears are measured with `SK_DPD_DEF`. A missing drawing is a month with no drawing. Utilisation is the balance over the limit, only where the limit is above zero. Minimum-payment features use only months where a minimum is recorded and above zero.

## Stage 4 and 5: Time and signal

Stage 1 settled the time question: no month at or after the application. The only time rule needed is the one of notebook 09: "at the application" means the latest row within three months.

Stage 5 looks at what card use says about default, development applicants only, and at whether any of it is new beside the features the model has. Each summary is compared with the model's card features from the payment table, and scored among applicants those features describe alike.

In [6]:
dev_cc = cc[cc['SK_ID_CURR'].isin(DEV_IDS)].sort_values([LOAN, MONTH])
by = dev_cc['SK_ID_CURR']
limit = dev_cc['AMT_CREDIT_LIMIT_ACTUAL'].where(dev_cc['AMT_CREDIT_LIMIT_ACTUAL'] > 0)
util = (dev_cc['AMT_BALANCE'] / limit).clip(lower=0)
min_due = dev_cc['AMT_INST_MIN_REGULARITY'].where(dev_cc['AMT_INST_MIN_REGULARITY'] > 0)
paid = dev_cc['AMT_PAYMENT_TOTAL_CURRENT']
recent = dev_cc[MONTH] >= -12
latest = dev_cc.groupby(LOAN).tail(1)
latest = latest[latest[MONTH] >= -3]
summaries = {
    'average utilisation': util.groupby(by).mean(),
    'highest utilisation': util.groupby(by).max(),
    'utilisation in the latest month': (latest['AMT_BALANCE'] / latest['AMT_CREDIT_LIMIT_ACTUAL'].where(latest['AMT_CREDIT_LIMIT_ACTUAL'] > 0)).clip(lower=0).groupby(latest['SK_ID_CURR']).max(),
    'average utilisation, last 12 months': util.where(recent).groupby(by).mean(),
    'share of months over the limit': (util > 1).groupby(by).mean(),
    'share of months with a balance': (dev_cc['AMT_BALANCE'] > 0).groupby(by).mean(),
    'share of months with an ATM drawing': (dev_cc['AMT_DRAWINGS_ATM_CURRENT'].fillna(0) > 0).groupby(by).mean(),
    'ATM drawings over the limit, average': (dev_cc['AMT_DRAWINGS_ATM_CURRENT'].fillna(0) / limit).groupby(by).mean(),
    'share of due months paid under the minimum': (paid < min_due - 1).where(min_due.notna()).groupby(by).mean(),
    'share of due months paid nothing': (paid == 0).where(min_due.notna()).groupby(by).mean(),
    'payment over balance, average': (paid / dev_cc['AMT_BALANCE'].where(dev_cc['AMT_BALANCE'] > 0)).clip(upper=1).groupby(by).mean(),
    'share of months in arrears': (dev_cc['SK_DPD_DEF'] > 0).groupby(by).mean(),
    'latest limit': latest.groupby('SK_ID_CURR')['AMT_CREDIT_LIMIT_ACTUAL'].max(),
    'months on record': dev_cc.groupby('SK_ID_CURR').size(),
}
inst_dev = pd.read_parquet(f'{PROCESSED}/installments_train_features.parquet').loc[DEV_IDS]
prev_dev = pd.read_parquet(f'{PROCESSED}/previous_train_features.parquet').loc[DEV_IDS]
model_meta = json.loads(open('../artifacts/model_with_installments/metadata.json', encoding='utf-8').read())
in_model = pd.concat([inst_dev, prev_dev], axis=1)[[c for c in model_meta['features'] if c.startswith(('INST_', 'PREV_'))]]
strata = pd.cut(inst_dev['INST_CARD_N_INSTALMENTS'], [-1, 0, 12, 36, np.inf], labels=False).astype(str) + '_' + (inst_dev['INST_CARD_SHARE_LATE_5D'].fillna(0) > 0).astype(str) + '_' + \
         pd.qcut(inst_dev['INST_CARD_MEAN_PAYMENT_12M'].fillna(-1), 4, labels=False, duplicates='drop').astype(str)
rows = {}
for name, values in summaries.items():
    v = values.reindex(DEV_IDS)
    ok = v.notna().values
    a = roc_auc_score(y_dev.values[ok], v.values[ok])
    direction = a > 0.5
    inside, weight = [], []
    for s in strata.unique():
        m = (strata == s).values & ok
        if m.sum() > 1000 and 0 < y_dev.values[m].sum() < m.sum() and v.values[m].std() > 0:
            b = roc_auc_score(y_dev.values[m], v.values[m])
            inside.append(b if direction else 1 - b)
            weight.append(m.sum())
    corr = in_model.corrwith(v, method='spearman').abs()
    rows[name] = {'card holders with a value': int(ok.sum()), 'auc_alone': max(a, 1 - a), 'higher_means': 'more risk' if direction else 'less risk',
                  'within groups alike on the model\'s card features': np.average(inside, weights=weight) if inside else np.nan, 'closest model feature': corr.idxmax(), 'its correlation': corr.max()}
signal = pd.DataFrame(rows).T.sort_values('auc_alone', ascending=False)
print(signal.to_string(float_format=lambda v: f'{v:,.3f}'))

def bands(values, edges, labels, title):
    print(f"\n{title}")
    print(rate_table(pd.cut(values.reindex(DEV_IDS), edges, labels=labels)).to_string(float_format=lambda v: f'{v:,.2f}'))
bands(summaries['average utilisation'], [-0.01, 0.0001, 0.3, 0.6, 0.9, np.inf], ['none', 'up to 30%', '30 to 60%', '60 to 90%', 'over 90%'], 'card holders by average utilisation:')
bands(summaries['share of months with an ATM drawing'], [-0.01, 0, 0.1, 0.3, 1], ['never', 'up to 10% of months', '10 to 30%', 'over 30%'], 'by share of months with a cash withdrawal:')
bands(summaries['share of due months paid under the minimum'], [-0.01, 0, 0.1, 0.3, 1], ['never', 'up to 10%', '10 to 30%', 'over 30%'], 'by share of months paid under the minimum, where one was due:')
bands(summaries['utilisation in the latest month'], [-0.01, 0.0001, 0.5, 0.9, 1.0, np.inf], ['none', 'up to 50%', '50 to 90%', '90 to 100%', 'over the limit'], 'card holders with a card in use at the application, by its utilisation then:')

                                           card holders with a value auc_alone higher_means within groups alike on the model's card features       closest model feature its correlation
utilisation in the latest month                                52026     0.652    more risk                                            0.590        INST_CARD_ACTIVE_90D           0.655
average utilisation, last 12 months                            57214     0.639    more risk                                            0.577        INST_CARD_ACTIVE_90D           0.695
average utilisation                                            68979     0.631    more risk                                            0.574        INST_CARD_ACTIVE_90D           0.684
share of months over the limit                                 69683     0.619    more risk                                            0.577     INST_CARD_N_INSTALMENTS           0.507
share of months with a balance                                 69683     0.

                applicants  default_pct  pct_of_applicants  standard_error  ratio_to_average
none                 27523         5.70              11.19            0.14              0.71
up to 50%             5090         6.52               2.07            0.35              0.81
50 to 90%             5953         9.39               2.42            0.38              1.16
90 to 100%            6655        14.23               2.71            0.43              1.76
over the limit        6805        20.26               2.77            0.49              2.51


## Stage 6: Aggregation design

| Rule | From |
|---|---|
| Utilisation is the balance over the limit, only where the limit is above zero, and never below zero | Stage 3 |
| A missing drawing amount is a month with no drawing | Stage 3 |
| Minimum-payment features use months where a minimum above 1 is recorded. A month is under the minimum if less than the minimum minus 1 was paid | Stage 3; and stage 7, where a consistency rule found that a minimum of 1 or less made the two definitions disagree |
| Arrears use `SK_DPD_DEF` | Stage 3, as in notebook 09 |
| "At the application" is each card's latest row, if within 3 months | Stage 1, as in notebook 09 |
| "Recent" is the last 12 months | As before |
| Payment over balance is capped at 1 | A payment can exceed the balance when a card is paid off |
| Applicants with no card have a card count of 0 and missing values everywhere else | As before |

**The features**, 22, prefixed `CC_` in the saved table.

| Family | Features |
|---|---|
| Volume | `N_CARDS`, `MONTHS_BACK`, `MONTHS_SINCE_LATEST_ROW` |
| Utilisation | `UTIL_MEAN`, `UTIL_MAX`, `UTIL_LATEST`, `UTIL_MEAN_12M`, `SHARE_OVER_LIMIT`, `SHARE_WITH_BALANCE`, `BALANCE_LATEST` |
| Limit | `LIMIT_LATEST`, `LIMIT_GROWTH` (latest limit over the first limit above zero) |
| Cash | `SHARE_ATM`, `ATM_TO_LIMIT_MEAN`, `N_ATM_12M` (number of ATM drawings in the last 12 months) |
| Payments | `SHARE_UNDER_MIN`, `SHARE_PAID_NOTHING`, `PAY_TO_BALANCE_MEAN`, `PAY_TO_MIN_MEAN` (capped at 5) |
| Arrears | `SHARE_ARREARS`, `MAX_DPD`, `N_DEMAND` (cards ever in status `Demand`) |

In [7]:
RECENT_MONTHS, LATEST_WINDOW, PAY_TO_MIN_CAP = 12, 3, 5
MIN_DUE_ABOVE = 1          # a month counts as due when the minimum is above 1; smaller minimums are residue (1,469 of 1.6 million due months)
FEATURE_FAMILY = {
    'N_CARDS': 'volume', 'MONTHS_BACK': 'volume', 'MONTHS_SINCE_LATEST_ROW': 'volume',
    'UTIL_MEAN': 'utilisation', 'UTIL_MAX': 'utilisation', 'UTIL_LATEST': 'utilisation', 'UTIL_MEAN_12M': 'utilisation', 'SHARE_OVER_LIMIT': 'utilisation',
    'SHARE_WITH_BALANCE': 'utilisation', 'BALANCE_LATEST': 'utilisation',
    'LIMIT_LATEST': 'limit', 'LIMIT_GROWTH': 'limit',
    'SHARE_ATM': 'cash', 'ATM_TO_LIMIT_MEAN': 'cash', 'N_ATM_12M': 'cash',
    'SHARE_UNDER_MIN': 'payments', 'SHARE_PAID_NOTHING': 'payments', 'PAY_TO_BALANCE_MEAN': 'payments', 'PAY_TO_MIN_MEAN': 'payments',
    'SHARE_ARREARS': 'arrears', 'MAX_DPD': 'arrears', 'N_DEMAND': 'arrears',
}
FEATURES = list(FEATURE_FAMILY)
assert len(FEATURES) == 22

def prepare_cards(raw):
    """Check the time rule and sort. No row is removed."""
    assert (raw['MONTHS_BALANCE'] < 0).all(), 'a month on or after the application'
    return raw.sort_values(['SK_ID_PREV', 'MONTHS_BALANCE'], kind='mergesort').reset_index(drop=True)

def build_card_features(prepared, ids):
    """One row per applicant in `ids`, from prepared monthly card rows. Own rows only, no outcome, no number estimated from the data."""
    p = prepared
    who, month = p['SK_ID_CURR'], p['MONTHS_BALANCE']
    limit = p['AMT_CREDIT_LIMIT_ACTUAL'].where(p['AMT_CREDIT_LIMIT_ACTUAL'] > 0)
    balance = p['AMT_BALANCE']
    util = (balance / limit).clip(lower=0)
    atm = p['AMT_DRAWINGS_ATM_CURRENT'].fillna(0)
    atm_count = p['CNT_DRAWINGS_ATM_CURRENT'].fillna(0)
    paid = p['AMT_PAYMENT_TOTAL_CURRENT']
    min_due = p['AMT_INST_MIN_REGULARITY'].where(p['AMT_INST_MIN_REGULARITY'] > MIN_DUE_ABOVE)
    recent = month >= -RECENT_MONTHS
    mean = lambda values: values.groupby(who).mean()

    last = p.groupby('SK_ID_PREV', sort=False).tail(1)
    last = last[last['MONTHS_BALANCE'] >= -LATEST_WINDOW]
    last_limit = last['AMT_CREDIT_LIMIT_ACTUAL'].where(last['AMT_CREDIT_LIMIT_ACTUAL'] > 0)
    first_limit = p.loc[p['AMT_CREDIT_LIMIT_ACTUAL'] > 0].groupby('SK_ID_PREV', sort=False).agg(applicant=('SK_ID_CURR', 'first'), first=('AMT_CREDIT_LIMIT_ACTUAL', 'first'), last=('AMT_CREDIT_LIMIT_ACTUAL', 'last'))

    f = pd.DataFrame(index=pd.Index(sorted(who.unique()), name='SK_ID_CURR'))
    f['N_CARDS'] = p.groupby('SK_ID_CURR')['SK_ID_PREV'].nunique()
    f['MONTHS_BACK'] = -month.groupby(who).min()
    f['MONTHS_SINCE_LATEST_ROW'] = -month.groupby(who).max()
    f['UTIL_MEAN'] = mean(util)
    f['UTIL_MAX'] = util.groupby(who).max()
    f['UTIL_LATEST'] = (last['AMT_BALANCE'] / last_limit).clip(lower=0).groupby(last['SK_ID_CURR']).max()
    f['UTIL_MEAN_12M'] = mean(util.where(recent))
    f['SHARE_OVER_LIMIT'] = mean((util > 1).astype(float).where(util.notna()))
    f['SHARE_WITH_BALANCE'] = mean((balance > 0).astype(float))
    f['BALANCE_LATEST'] = last.groupby('SK_ID_CURR')['AMT_BALANCE'].sum()
    f['LIMIT_LATEST'] = last.groupby('SK_ID_CURR')['AMT_CREDIT_LIMIT_ACTUAL'].sum()
    f['LIMIT_GROWTH'] = (first_limit['last'] / first_limit['first']).groupby(first_limit['applicant']).max()
    f['SHARE_ATM'] = mean((atm > 0).astype(float))
    f['ATM_TO_LIMIT_MEAN'] = mean(atm / limit)
    f['N_ATM_12M'] = atm_count.where(recent, 0).groupby(who).sum()
    f['SHARE_UNDER_MIN'] = mean((paid < min_due - 1).astype(float).where(min_due.notna()))
    f['SHARE_PAID_NOTHING'] = mean((paid == 0).astype(float).where(min_due.notna()))
    f['PAY_TO_BALANCE_MEAN'] = mean((paid / balance.where(balance > 0)).clip(upper=1))
    f['PAY_TO_MIN_MEAN'] = mean((paid / min_due).clip(upper=PAY_TO_MIN_CAP))
    f['SHARE_ARREARS'] = mean((p['SK_DPD_DEF'] > 0).astype(float))
    f['MAX_DPD'] = p['SK_DPD_DEF'].groupby(who).max()
    f['N_DEMAND'] = (p['NAME_CONTRACT_STATUS'] == 'Demand').groupby(p['SK_ID_PREV'], sort=False).any().groupby(p.groupby('SK_ID_PREV', sort=False)['SK_ID_CURR'].first()).sum()
    out = f[FEATURES].astype(float).reindex(pd.Index(ids, name='SK_ID_CURR'))
    out['N_CARDS'] = out['N_CARDS'].fillna(0)
    return out

for name in ['dev_cc', 'by', 'limit', 'util', 'min_due', 'paid', 'recent', 'latest', 'prev', 'prev_ids', 'approved', 'known', 'miss', 'active', 'dpd', 'dpd_def', 'active2', 'has1', 'active5']:
    globals().pop(name, None)
gc.collect()
ALL_IDS = TRAIN_IDS.append(TEST_IDS)
started = time.time()
features = build_card_features(prepare_cards(cc), ALL_IDS)
no_card = features['N_CARDS'] == 0
print(f"built in {time.time() - started:.0f} seconds: {features.shape}; applicants with no card {int(no_card.sum()):,}; every other feature missing for them: {bool(features.loc[no_card, FEATURES[1:]].isna().all().all())}; infinite values: {int(np.isinf(features.to_numpy()).sum())}")
assert not np.isinf(features.to_numpy()).any()

dev_features, train_f, test_f = features.reindex(DEV_IDS), features.reindex(TRAIN_IDS), features.reindex(TEST_IDS)
rows = {}
for name in FEATURES:
    v = dev_features[name]
    ok = v.notna().values
    a = roc_auc_score(y_dev.values[ok], v.values[ok]) if v[ok].nunique() > 1 else 0.5
    rows[name] = {'family': FEATURE_FAMILY[name], 'has_a_value_train_pct': train_f[name].notna().mean() * 100, 'has_a_value_test_pct': test_f[name].notna().mean() * 100,
                  'zero_pct_of_those': (train_f[name] == 0).sum() / max(train_f[name].notna().sum(), 1) * 100, 'median': train_f[name].median(), 'auc_alone': max(a, 1 - a),
                  'higher_means': 'more risk' if a > 0.5 else 'less risk', 'mean_train': train_f[name].mean(), 'mean_test': test_f[name].mean()}
print(pd.DataFrame(rows).T.to_string(float_format=lambda v: f'{v:,.3f}'))

built in 11 seconds: (356255, 22); applicants with no card 252,697; every other feature missing for them: True; infinite values: 0


                              family has_a_value_train_pct has_a_value_test_pct zero_pct_of_those      median auc_alone higher_means  mean_train   mean_test
N_CARDS                       volume               100.000              100.000            71.739       0.000     0.512    more risk       0.284       0.346
MONTHS_BACK                   volume                28.261               34.164             0.000      22.000     0.558    less risk      37.585      37.274
MONTHS_SINCE_LATEST_ROW       volume                28.261               34.164             0.000       1.000     0.519    less risk       1.498       1.628
UTIL_MEAN                utilisation                27.978               33.664            30.917       0.253     0.631    more risk       0.327       0.287
UTIL_MAX                 utilisation                27.978               33.664            30.917       0.953     0.611    more risk       0.631       0.576
UTIL_LATEST              utilisation                21.075

## Stage 7: Verification

As in notebooks 07 and 09: one pass from the raw file, an independent rebuild with plain loops, rules every row must obey, own rows only, and the finished tables. No target is used.

A note on the first run. The rule that paying nothing is never more common than paying under the minimum failed. The cause was 1,469 months (0.09% of months with a minimum) with a minimum of 1 or less: paying nothing in such a month is not "under the minimum" once a tolerance of 1 is allowed. The features were redefined to count a month as due only when the minimum is above 1, in the column-wise function and in the loop alike, and the rule now holds by construction.

In [8]:
stage6 = features.copy()
for name in ['features', 'dev_features', 'train_f', 'test_f', 'inst_dev', 'prev_dev', 'in_model', 'strata']:
    globals().pop(name, None)
gc.collect()
raw = pd.read_csv(f'{RAW}/credit_card_balance.csv')
features = build_card_features(prepare_cards(raw), ALL_IDS)

def same_table(a, b, tolerance=1e-9):
    if not (a.index.equals(b.index) and list(a.columns) == list(b.columns)):
        return False
    x, y = a.to_numpy(dtype=float), b.to_numpy(dtype=float)
    return bool(((np.isnan(x) & np.isnan(y)) | (np.abs(x - y) <= tolerance * np.maximum(1, np.abs(y)))).all())
print(f"from the raw file, identical to stage 6: {same_table(features, stage6)}")
assert same_table(features, stage6)
del stage6

def slow_features(rows):
    """The 22 features of one applicant from raw rows, as plain records. Written independently of build_card_features."""
    nan = float('nan')
    ok = lambda v: v == v and v is not None
    mean = lambda vals: sum(vals) / len(vals) if vals else nan
    cards = {}
    for r in sorted(rows, key=lambda r: (r['SK_ID_PREV'], r['MONTHS_BALANCE'])):
        cards.setdefault(r['SK_ID_PREV'], []).append(r)
    allr = [r for months in cards.values() for r in months]
    f = {'N_CARDS': len(cards), 'MONTHS_BACK': -min(r['MONTHS_BALANCE'] for r in allr), 'MONTHS_SINCE_LATEST_ROW': -max(r['MONTHS_BALANCE'] for r in allr)}
    def util_of(r):
        return max(r['AMT_BALANCE'] / r['AMT_CREDIT_LIMIT_ACTUAL'], 0) if r['AMT_CREDIT_LIMIT_ACTUAL'] > 0 else None
    utils = [u for u in (util_of(r) for r in allr) if u is not None]
    f['UTIL_MEAN'] = mean(utils)
    f['UTIL_MAX'] = max(utils) if utils else nan
    lasts = [months[-1] for months in cards.values() if months[-1]['MONTHS_BALANCE'] >= -3]
    lu = [u for u in (util_of(r) for r in lasts) if u is not None]
    f['UTIL_LATEST'] = max(lu) if lu else nan
    f['UTIL_MEAN_12M'] = mean([u for u in (util_of(r) for r in allr if r['MONTHS_BALANCE'] >= -12) if u is not None])
    f['SHARE_OVER_LIMIT'] = mean([1.0 if u > 1 else 0.0 for u in utils])
    f['SHARE_WITH_BALANCE'] = mean([1.0 if r['AMT_BALANCE'] > 0 else 0.0 for r in allr])
    f['BALANCE_LATEST'] = sum(r['AMT_BALANCE'] for r in lasts) if lasts else nan
    f['LIMIT_LATEST'] = sum(r['AMT_CREDIT_LIMIT_ACTUAL'] for r in lasts) if lasts else nan
    growth = []
    for months in cards.values():
        positive = [r['AMT_CREDIT_LIMIT_ACTUAL'] for r in months if r['AMT_CREDIT_LIMIT_ACTUAL'] > 0]
        if positive:
            growth.append(positive[-1] / positive[0])
    f['LIMIT_GROWTH'] = max(growth) if growth else nan
    atm = lambda r: r['AMT_DRAWINGS_ATM_CURRENT'] if ok(r['AMT_DRAWINGS_ATM_CURRENT']) else 0.0
    f['SHARE_ATM'] = mean([1.0 if atm(r) > 0 else 0.0 for r in allr])
    f['ATM_TO_LIMIT_MEAN'] = mean([atm(r) / r['AMT_CREDIT_LIMIT_ACTUAL'] for r in allr if r['AMT_CREDIT_LIMIT_ACTUAL'] > 0])
    f['N_ATM_12M'] = sum((r['CNT_DRAWINGS_ATM_CURRENT'] if ok(r['CNT_DRAWINGS_ATM_CURRENT']) else 0.0) for r in allr if r['MONTHS_BALANCE'] >= -12)
    due = [r for r in allr if ok(r['AMT_INST_MIN_REGULARITY']) and r['AMT_INST_MIN_REGULARITY'] > 1]
    f['SHARE_UNDER_MIN'] = mean([1.0 if r['AMT_PAYMENT_TOTAL_CURRENT'] < r['AMT_INST_MIN_REGULARITY'] - 1 else 0.0 for r in due])
    f['SHARE_PAID_NOTHING'] = mean([1.0 if r['AMT_PAYMENT_TOTAL_CURRENT'] == 0 else 0.0 for r in due])
    f['PAY_TO_BALANCE_MEAN'] = mean([min(r['AMT_PAYMENT_TOTAL_CURRENT'] / r['AMT_BALANCE'], 1.0) for r in allr if r['AMT_BALANCE'] > 0])
    f['PAY_TO_MIN_MEAN'] = mean([min(r['AMT_PAYMENT_TOTAL_CURRENT'] / r['AMT_INST_MIN_REGULARITY'], 5.0) for r in due])
    f['SHARE_ARREARS'] = mean([1.0 if r['SK_DPD_DEF'] > 0 else 0.0 for r in allr])
    f['MAX_DPD'] = max(r['SK_DPD_DEF'] for r in allr)
    f['N_DEMAND'] = sum(1 for months in cards.values() if any(r['NAME_CONTRACT_STATUS'] == 'Demand' for r in months))
    return f

rng = np.random.default_rng(SEED)
with_card = features.index[features['N_CARDS'] > 0]
picks = {'at random': rng.choice(with_card, 300, replace=False)}
for label, mask in [('more than one card', features['N_CARDS'] > 1), ('over the limit at some time', features['SHARE_OVER_LIMIT'] > 0), ('in arrears', features['SHARE_ARREARS'] > 0),
                    ('a card in Demand', features['N_DEMAND'] > 0), ('paid nothing when due', features['SHARE_PAID_NOTHING'] > 0), ('no limit ever', features['UTIL_MEAN'].isna() & (features['N_CARDS'] > 0))]:
    picks[label] = rng.choice(features.index[mask], min(40, int(mask.sum())), replace=False)
sample_ids = pd.Index(np.unique(np.concatenate(list(picks.values()))))
sample_rows = raw[raw['SK_ID_CURR'].isin(sample_ids)]
slow = pd.DataFrame({a: slow_features(g.to_dict('records')) for a, g in sample_rows.groupby('SK_ID_CURR')}).T[FEATURES].astype(float)
x, y = features.loc[slow.index].to_numpy(), slow.to_numpy()
agree = (np.isnan(x) & np.isnan(y)) | (np.abs(x - y) <= 1e-9 * np.maximum(1, np.abs(y)))
print('sample:', {k: len(v) for k, v in picks.items()}, f'-> {len(sample_ids)} applicants, {len(sample_rows):,} rows')
print(f"cells compared {agree.size:,}; in agreement {int(agree.sum()):,}; features with any disagreement: {[FEATURES[j] for j in np.where(~agree.all(axis=0))[0]]}")
if not agree.all():
    j = int(np.where(~agree.all(axis=0))[0][0]); i = int(np.where(~agree[:, j])[0][0])
    print(f"first disagreement: {FEATURES[j]} for applicant {slow.index[i]}: columns {x[i, j]}, loops {y[i, j]}")
assert agree.all()

f = features
has = f['N_CARDS'] > 0
shares = ['SHARE_OVER_LIMIT', 'SHARE_WITH_BALANCE', 'SHARE_ATM', 'SHARE_UNDER_MIN', 'SHARE_PAID_NOTHING', 'SHARE_ARREARS', 'PAY_TO_BALANCE_MEAN']
rules = {
    'no infinite value': not np.isinf(f.to_numpy()).any(),
    'every share is between 0 and 1': all(f[c].dropna().between(0, 1).all() for c in shares),
    'no card means a count of 0 and nothing else': bool(f.loc[~has, FEATURES[1:]].isna().all().all()),
    'utilisation is never negative, and the mean never exceeds the maximum': bool((f[['UTIL_MEAN', 'UTIL_MAX', 'UTIL_LATEST', 'UTIL_MEAN_12M']].min().min() >= 0) and (f['UTIL_MEAN'] <= f['UTIL_MAX'] + 1e-12)[f['UTIL_MEAN'].notna()].all()),
    'over the limit at some time exactly when the highest utilisation is above 1': bool(((f['SHARE_OVER_LIMIT'] > 0) == (f['UTIL_MAX'] > 1))[f['UTIL_MAX'].notna()].all()),
    'paid nothing never more often than paid under the minimum': bool((f['SHARE_PAID_NOTHING'] <= f['SHARE_UNDER_MIN'] + 1e-12)[f['SHARE_UNDER_MIN'].notna()].all()),
    'payment over minimum between 0 and the cap': bool(f['PAY_TO_MIN_MEAN'].dropna().between(0, PAY_TO_MIN_CAP).all()),
    'latest balance and limit present exactly when a card has a recent row': bool((f['LIMIT_LATEST'].notna() == (f['MONTHS_SINCE_LATEST_ROW'] <= LATEST_WINDOW)).all()),
    'counts are whole numbers': all((f[c].dropna() % 1 == 0).all() for c in ['N_CARDS', 'N_ATM_12M', 'N_DEMAND', 'MAX_DPD']),
    'the record goes back 1 to 96 months': bool(f['MONTHS_BACK'].dropna().between(1, 96).all()),
}
for name, ok_rule in rules.items():
    print(f"{'PASS' if ok_rule else 'FAIL'}  {name}")
assert all(rules.values())

subset_ids = pd.Index(rng.choice(with_card, 3000, replace=False))
subset_rows = raw[raw['SK_ID_CURR'].isin(subset_ids)]
alone = build_card_features(prepare_cards(subset_rows), subset_ids)
half_a, half_b = subset_ids[:1500], subset_ids[1500:]
tampered_rows = subset_rows.copy()
tampered_rows.loc[tampered_rows['SK_ID_CURR'].isin(half_a), 'AMT_BALANCE'] *= 2
tampered = build_card_features(prepare_cards(tampered_rows), subset_ids)
own = {
    'built from their own rows alone: the same as the full build': same_table(alone, features.loc[subset_ids]),
    'rows shuffled: the same': same_table(build_card_features(prepare_cards(subset_rows.sample(frac=1, random_state=SEED)), subset_ids), alone),
    'other applicants changed: untouched applicants are the same': same_table(tampered.loc[half_b], alone.loc[half_b]),
    'and the changed applicants did change': not same_table(tampered.loc[half_a], alone.loc[half_a]),
}
for name, ok_rule in own.items():
    print(f"{'PASS' if ok_rule else 'FAIL'}  {name}")
assert all(own.values())
app_train_index = pd.read_parquet(f'{PROCESSED}/application_train_features.parquet', columns=[]).index
app_test_index = pd.read_parquet(f'{PROCESSED}/application_test_features.parquet', columns=[]).index
PREFIX = 'CC_'
train_table, test_table = features.reindex(app_train_index).add_prefix(PREFIX), features.reindex(app_test_index).add_prefix(PREFIX)
assert train_table.index.equals(app_train_index) and test_table.index.equals(app_test_index) and train_table.shape[1] == 22 and 'TARGET' not in train_table.columns
print('PASS  finished tables: 307,511 and 48,744 rows in application order, 22 numeric CC_ columns, no outcome')

from the raw file, identical to stage 6: True


sample: {'at random': 300, 'more than one card': 40, 'over the limit at some time': 40, 'in arrears': 40, 'a card in Demand': 22, 'paid nothing when due': 40, 'no limit ever': 40} -> 521 applicants, 24,466 rows
cells compared 11,462; in agreement 11,462; features with any disagreement: []


PASS  no infinite value
PASS  every share is between 0 and 1
PASS  no card means a count of 0 and nothing else
PASS  utilisation is never negative, and the mean never exceeds the maximum
PASS  over the limit at some time exactly when the highest utilisation is above 1
PASS  paid nothing never more often than paid under the minimum
PASS  payment over minimum between 0 and the cap
PASS  latest balance and limit present exactly when a card has a recent row
PASS  counts are whole numbers
PASS  the record goes back 1 to 96 months


PASS  built from their own rows alone: the same as the full build
PASS  rows shuffled: the same
PASS  other applicants changed: untouched applicants are the same
PASS  and the changed applicants did change
PASS  finished tables: 307,511 and 48,744 rows in application order, 22 numeric CC_ columns, no outcome



### Stage 4 to 7 findings

**How a card is used says a great deal, and most of it is new to the model**

| Summary | Card holders | AUC alone | Within groups alike on the model's card features |
|---|---:|---:|---:|
| Utilisation in the latest month | 52,026 | 0.652 | 0.590 |
| Average utilisation, last 12 months | 57,214 | 0.639 | 0.577 |
| Average utilisation | 68,979 | 0.631 | 0.574 |
| Share of months over the limit | 69,683 | 0.619 | 0.577 |
| Share of months with a cash withdrawal | 69,683 | 0.615 | 0.559 |
| Payment over balance | 47,737 | 0.588 | 0.565 |
| Share of due months paid under the minimum | 47,506 | 0.501 | 0.504 |
| Share of months in arrears | 69,683 | 0.506 | 0.505 |

- These are the strongest summaries any history table in this project has produced, and they keep most of their strength among card holders the model's existing card features describe alike. Notebook 09's arrears fell from 0.526 to 0.508 on the same kind of test.

| Utilisation in the latest month | Card holders | Default rate |
|---|---:|---:|
| No balance | 27,523 | 5.7% |
| Up to 50% | 5,090 | 6.5% |
| 50 to 90% | 5,953 | 9.4% |
| 90 to 100% | 6,655 | 14.2% |
| Over the limit | 6,805 | 20.3% |

- A card used past its limit at the time of the application goes with a default rate of 20%, three and a half times the rate of a card with no balance. Cash withdrawals tell the same story: 5.9% default for card holders who never withdraw cash, 14.8% for those who do so in more than 30% of months.
- **Missed minimum payments and arrears say nothing**, both alone (0.501 and 0.506) and beside the model. Card holders who never paid under the minimum default at 10.2%, more than those who sometimes did. The cards that run into arrears belong to applicants the model already recognises as risky through the payment table.
- So what this table adds is not conduct, which the payment table had, but **exposure**: how much of the available credit the applicant is using, and whether they use it for cash.

**22 candidate features**, built in 11 seconds for all applicants. 71.7% of applicants have no card and every feature but the count missing.

**Verification**
- From the raw file in one pass: identical to stage 6. An independent rebuild with plain loops, for 521 applicants weighted towards the awkward cases: all 11,462 cells agree. Ten consistency rules hold, and an applicant's features depend on their own rows only.
- **One correction.** On the first run the rule that paying nothing is never more common than paying under the minimum failed. 1,469 months, 0.09% of due months, have a minimum of 1 or less, and with a tolerance of 1 paying nothing in such a month is not "under the minimum". A month now counts as due only when the minimum is above 1, in the features and in the loop alike. Like the two corrections in notebooks 07 and 09, the feature code and the check disagreed because a definition said less than it seemed to.


## Stage 8: Screening

The three rules of notebooks 03, 05, 07 and 09, unchanged, with the two additions of notebook 07: redundancy is checked against the features already saved from earlier tables as well as among candidates, and "beyond the current model" means beyond the model saved by notebook 08, with 108 features from four tables.

| Rule | Uses the outcome? | Action |
|---|---|---|
| 1. Shift: PSI of 0.10 or more, a shift AUC of 0.55 or more, or for a rare event a test rate outside 0.67 to 1.5 times the train rate | No | Flag; drop if caused by how the data was supplied, keep with a warning if the applicants differ |
| 2. Redundancy: rank correlation of 0.90 or more with another candidate or a saved feature | No | Keep one of each group of candidates; drop a candidate that repeats a saved feature |
| 3. No signal: alone under 0.51, and an interval beyond the model that includes 0.50, and a flat risk ratio for a rare event | Yes | Drop only if all three hold |

| Section | What it does |
|---|---|
| 8.1 | Shift |
| 8.2 | Redundancy |
| 8.3 | Signal alone |
| 8.4 | Signal beyond the current model |
| 8.5 | The rules applied |

In [9]:
import json, warnings
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from lightgbm import LGBMClassifier

for name in ['raw', 'sample_rows', 'subset_rows', 'tampered_rows', 'alone', 'tampered', 'slow', 'cc', 'cc_part', 'cards']:
    globals().pop(name, None)
gc.collect()

PF = features
FAMILY_OF = FEATURE_FAMILY
train_h = TRAIN_IDS[(PF.loc[TRAIN_IDS, 'N_CARDS'] > 0).values]
test_h = TEST_IDS[(PF.loc[TEST_IDS, 'N_CARDS'] > 0).values]
A_, B_ = PF.loc[train_h], PF.loc[test_h]
both_h = train_h.append(test_h)
is_test_h = np.r_[np.zeros(len(train_h)), np.ones(len(test_h))]

def psi_feature(a, b, bins=10):
    if a.nunique() <= 12:
        code_of = lambda s: s.fillna(-1).values
    else:
        edges = np.unique(np.nanquantile(a.values, np.linspace(0, 1, bins + 1)[1:-1]))
        code_of = lambda s: np.where(s.isna(), -1, np.digitize(s.values, edges, right=True))
    shares = pd.DataFrame({'a': pd.Series(code_of(a)).value_counts(normalize=True), 'b': pd.Series(code_of(b)).value_counts(normalize=True)}).fillna(0) + 1e-4
    return float(((shares['b'] - shares['a']) * np.log(shares['b'] / shares['a'])).sum())

def joint_shift(X, max_iter=150):
    scores = []
    for fit_idx, val_idx in StratifiedKFold(3, shuffle=True, random_state=SEED).split(X, is_test_h):
        m = HistGradientBoostingClassifier(max_iter=max_iter, random_state=SEED).fit(X[fit_idx], is_test_h[fit_idx])
        scores.append(roc_auc_score(is_test_h[val_idx], m.predict_proba(X[val_idx])[:, 1]))
    return float(np.mean(scores))

shift_table = {}
for c in FEATURES:
    ok = PF.loc[both_h, c].notna().values
    a = roc_auc_score(is_test_h[ok], PF.loc[both_h, c].values[ok])
    mostly_zero = (A_[c] == 0).sum() / A_[c].notna().sum() >= 0.70
    rate_a, rate_b = (A_[c] > 0).sum() / A_[c].notna().sum(), (B_[c] > 0).sum() / B_[c].notna().sum()
    shift_table[c] = {'family': FAMILY_OF[c], 'psi': psi_feature(A_[c], B_[c]), 'shift_auc': max(a, 1 - a), 'flexible_shift_auc': joint_shift(PF.loc[both_h, [c]].to_numpy(), 60),
                      'mostly_zero': mostly_zero, 'above_zero_train_pct': rate_a * 100 if mostly_zero else np.nan, 'above_zero_test_pct': rate_b * 100 if mostly_zero else np.nan,
                      'test_to_train_rate': rate_b / rate_a if mostly_zero else np.nan}
shift_table = pd.DataFrame(shift_table).T
shift_table['flagged'] = (shift_table['psi'].astype(float) >= 0.10) | (shift_table['shift_auc'].astype(float) >= 0.55) | \
                         (shift_table['test_to_train_rate'].astype(float) < 0.67) | (shift_table['test_to_train_rate'].astype(float) > 1.5)
print(f"applicants with a card: train {len(train_h):,}, test {len(test_h):,}\n")
print(shift_table.sort_values('psi', ascending=False).to_string(float_format=lambda v: f'{v:,.3f}'))
SHIFT_FLAGGED = list(shift_table.index[shift_table['flagged']])
FINE_DETAIL = shift_table.index[(shift_table['flexible_shift_auc'].astype(float) >= 0.55) & ~shift_table['flagged']].tolist()
print(f"\nflagged by rule 1: {SHIFT_FLAGGED}; fine-detail warnings: {FINE_DETAIL}")
JOINT_ALL = joint_shift(PF.loc[both_h].to_numpy())
print(f"a model given all 22 features, telling test applicants from train applicants: AUC {JOINT_ALL:.4f} (the payment features of notebook 07: 0.629; the monthly balances of notebook 09: 0.643)")

applicants with a card: train 86,905, test 16,653

                              family   psi shift_auc flexible_shift_auc mostly_zero above_zero_train_pct above_zero_test_pct test_to_train_rate  flagged
MONTHS_BACK                   volume 0.198     0.532              0.667       False                  NaN                 NaN                NaN     True
LIMIT_LATEST                   limit 0.109     0.505              0.605       False                  NaN                 NaN                NaN     True
MONTHS_SINCE_LATEST_ROW       volume 0.097     0.518              0.549       False                  NaN                 NaN                NaN    False
BALANCE_LATEST           utilisation 0.091     0.504              0.549       False                  NaN                 NaN                NaN    False
SHARE_ATM                       cash 0.031     0.542              0.552       False                  NaN                 NaN                NaN    False
ATM_TO_LIMIT_MEAN              

a model given all 22 features, telling test applicants from train applicants: AUC 0.7806 (the payment features of notebook 07: 0.629; the monthly balances of notebook 09: 0.643)


### 8.2 Redundancy

In [10]:
rec_all = PF.loc[both_h]
corr = rec_all.corr(method='spearman')
pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack().dropna()
high = pairs[pairs.abs() >= 0.90].sort_values(key=np.abs, ascending=False)
print(f"pairs of candidates at 0.90 or more: {len(high)}")
print(high.round(3).to_string())
parent = {c: c for c in corr.index}
def find(c):
    while parent[c] != c:
        c = parent[c]
    return c
for (a, b), value in high.items():
    parent[find(a)] = find(b)
groups = {}
for c in corr.index:
    groups.setdefault(find(c), []).append(c)
groups = [m for m in groups.values() if len(m) > 1]
coverage = rec_all.notna().mean()
REDUNDANT_DROPPED, group_rows = [], []
for members in groups:
    ranked = sorted(members, key=lambda c: (-round(coverage[c], 3), shift_table.loc[c, 'psi']))
    REDUNDANT_DROPPED += ranked[1:]
    group_rows.append({'kept': ranked[0], 'dropped': ', '.join(ranked[1:])})
print(pd.DataFrame(group_rows).to_string(index=False) if group_rows else 'no groups')

saved = pd.read_parquet(f'{PROCESSED}/installments_train_features.parquet').join(pd.read_parquet(f'{PROCESSED}/previous_train_features.parquet')).join(pd.read_parquet(f'{PROCESSED}/pos_cash_train_features.parquet')).join(
    pd.read_parquet(f'{PROCESSED}/bureau_train_features.parquet')).join(pd.read_parquet(f'{PROCESSED}/application_train_features.parquet').drop(columns='TARGET').select_dtypes('number')).loc[train_h]
ranks_new, ranks_saved = PF.loc[train_h].rank(), saved.rank()
cross = pd.DataFrame({c: ranks_saved.corrwith(ranks_new[c]) for c in FEATURES})
closest = pd.DataFrame({'closest saved feature': cross.abs().idxmax(), 'rank correlation': [cross.loc[cross[c].abs().idxmax(), c] for c in FEATURES]}, index=FEATURES)
print(f"\nsaved features compared with: {saved.shape[1]}")
print(closest.sort_values('rank correlation', key=np.abs, ascending=False).to_string(float_format=lambda v: f'{v:.3f}'))
CROSS_REDUNDANT = {c: closest.loc[c, 'closest saved feature'] for c in FEATURES if abs(closest.loc[c, 'rank correlation']) >= 0.90}
print(f"\ncandidates that repeat a saved feature: {CROSS_REDUNDANT}")
del saved, ranks_new, ranks_saved
gc.collect()

pairs of candidates at 0.90 or more: 5
SHARE_ARREARS  MAX_DPD               0.994
UTIL_LATEST    BALANCE_LATEST        0.937
SHARE_ATM      ATM_TO_LIMIT_MEAN     0.922
UTIL_MEAN      SHARE_WITH_BALANCE    0.907
UTIL_LATEST    UTIL_MEAN_12M         0.905
              kept                    dropped
SHARE_WITH_BALANCE                  UTIL_MEAN
    BALANCE_LATEST UTIL_MEAN_12M, UTIL_LATEST
         SHARE_ATM          ATM_TO_LIMIT_MEAN
           MAX_DPD              SHARE_ARREARS



saved features compared with: 168
                                closest saved feature  rank correlation
UTIL_MAX                      INST_CARD_N_INSTALMENTS             0.751
SHARE_WITH_BALANCE               INST_CARD_ACTIVE_90D             0.719
UTIL_MEAN_12M                    INST_CARD_ACTIVE_90D             0.694
MONTHS_BACK                   INST_CARD_N_INSTALMENTS             0.693
UTIL_MEAN                        INST_CARD_ACTIVE_90D             0.684
UTIL_LATEST                      INST_CARD_ACTIVE_90D             0.654
SHARE_ATM                        INST_CARD_ACTIVE_90D             0.595
LIMIT_LATEST               INST_CARD_MEAN_PAYMENT_12M             0.592
ATM_TO_LIMIT_MEAN                INST_CARD_ACTIVE_90D             0.585
N_ATM_12M                  INST_CARD_MEAN_PAYMENT_12M             0.574
BALANCE_LATEST                   INST_CARD_ACTIVE_90D             0.570
SHARE_ARREARS                 INST_CARD_SHARE_LATE_5D             0.549
MAX_DPD                      

72

### 8.3 Signal alone, and 8.4 beyond the current model

As in notebook 07. The current model is rebuilt in a simple form from notebook 08's 108 features, fitted on four development folds to score the fifth, and applicants are cut into twenty groups by that score.

In [11]:
dev_h = DEV_IDS[(PF.loc[DEV_IDS, 'N_CARDS'] > 0).values]
D = PF.loc[dev_h]
y_h = y_dev.loc[dev_h].values
fold_h = split.loc[dev_h, 'fold'].values

def banded_auc(values, y, fold, bins=10):
    v = np.asarray(values, dtype=float)
    scores, per_fold = np.zeros(len(v)), []
    for k in range(5):
        fit, val = fold != k, fold == k
        edges = np.unique(np.nanquantile(v[fit], np.linspace(0, 1, bins + 1)[1:-1]))
        band = np.where(np.isnan(v), -1, np.digitize(v, edges, right=True))
        rates = pd.Series(y[fit]).groupby(band[fit]).mean()
        scores[val] = pd.Series(band[val]).map(rates).fillna(y[fit].mean()).values
        per_fold.append(roc_auc_score(y[val], scores[val]))
    return float(np.mean(per_fold))

alone_table = {}
for c in FEATURES:
    ok = D[c].notna().values
    plain = roc_auc_score(y_h[ok], D[c].values[ok])
    alone_table[c] = {'banded_auc': banded_auc(D[c].values, y_h, fold_h), 'plain_auc': max(plain, 1 - plain), 'higher_value_means': 'more risk' if plain > 0.5 else 'less risk'}
alone_table = pd.DataFrame(alone_table).T

current = json.loads(Path('../artifacts/model_with_installments/metadata.json').read_text(encoding='utf-8'))
MODEL_FEATURES, MODEL_TEXT = current['features'], current['text_feature_levels']
table = pd.read_parquet(f'{PROCESSED}/application_train_features.parquet')
for name in ['bureau', 'previous', 'installments']:
    table = table.join(pd.read_parquet(f'{PROCESSED}/{name}_train_features.parquet'))
table = table.loc[DEV_IDS, MODEL_FEATURES]
for c in MODEL_FEATURES:
    table[c] = pd.Categorical(table[c], categories=MODEL_TEXT[c]) if c in MODEL_TEXT else table[c].astype(float)
dev_fold_all = split.loc[DEV_IDS, 'fold'].values
started = time.time()
model_score = np.zeros(len(table))
for k in range(5):
    fit, val = dev_fold_all != k, dev_fold_all == k
    m = LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=10, min_child_samples=800, colsample_bytree=0.3, random_state=SEED, verbose=-1)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        m.fit(table.loc[fit], y_dev.values[fit])
        model_score[val] = m.predict_proba(table.loc[val])[:, 1]
print(f"current model ({len(MODEL_FEATURES)} features), simplified, out-of-fold: ROC AUC {roc_auc_score(y_dev.values, model_score):.4f} (notebook 08's full model: {current['evidence']['cross_validation']['roc_auc']:.4f}), {time.time() - started:.0f} seconds")
del table
score_h = pd.Series(model_score, index=DEV_IDS).loc[dev_h].values
group = pd.qcut(score_h, 20, labels=False)

def beyond(values, direction):
    v = np.asarray(values, dtype=float)
    inside, weight = [], []
    for g in range(20):
        m = (group == g) & ~np.isnan(v)
        if m.sum() < 200 or y_h[m].min() == y_h[m].max() or np.unique(v[m]).size < 2:
            continue
        a = roc_auc_score(y_h[m], v[m])
        inside.append(a if direction == 'more risk' else 1 - a)
        weight.append(m.sum())
    inside, weight = np.array(inside), np.array(weight)
    return float(np.average(inside, weights=weight)), float(inside.std(ddof=1) / np.sqrt(len(inside)))

def ratio_beyond(values):
    v = np.asarray(values, dtype=float)
    ratios, weight = [], []
    for g in range(20):
        m = (group == g) & ~np.isnan(v)
        with_event, without = y_h[m & (v > 0)], y_h[m & (v == 0)]
        if len(with_event) >= 30 and with_event.sum() > 0 and without.sum() > 0:
            ratios.append(with_event.mean() / without.mean())
            weight.append(len(with_event))
    return float(np.average(ratios, weights=weight)) if ratios else np.nan

def observed_over_expected(values):
    event = np.asarray(values, dtype=float) > 0
    expected = pd.Series(y_h).groupby(group).mean().reindex(group[event]).sum()
    return float(y_h[event].sum() / expected) if expected > 0 else np.nan

new = {}
for c in FEATURES:
    mean_b, se = beyond(D[c].values, alone_table.loc[c, 'higher_value_means'])
    mostly_zero = bool(shift_table.loc[c, 'mostly_zero'])
    new[c] = {'family': FAMILY_OF[c], 'alone': alone_table.loc[c, 'banded_auc'], 'beyond_model': mean_b, 'low': mean_b - 1.96 * se, 'high': mean_b + 1.96 * se,
              'risk_ratio_beyond_model': ratio_beyond(D[c].values) if mostly_zero else np.nan, 'observed_over_expected': observed_over_expected(D[c].values) if mostly_zero else np.nan,
              'applicants_with_the_event': int((D[c] > 0).sum()) if mostly_zero else np.nan}
new = pd.DataFrame(new).T
print()
print(new.sort_values('beyond_model', ascending=False).to_string(float_format=lambda v: f'{v:.3f}'))
print(f"\ninterval beyond the model wholly above 0.50: {int((new['low'] > 0.50).sum())} of {len(new)}")

current model (108 features), simplified, out-of-fold: ROC AUC 0.7875 (notebook 08's full model: 0.7932), 41 seconds



                              family alone beyond_model   low  high risk_ratio_beyond_model observed_over_expected applicants_with_the_event
UTIL_LATEST              utilisation 0.633        0.551 0.533 0.570                     NaN                    NaN                       NaN
ATM_TO_LIMIT_MEAN               cash 0.613        0.547 0.537 0.557                     NaN                    NaN                       NaN
SHARE_ATM                       cash 0.612        0.544 0.534 0.554                     NaN                    NaN                       NaN
UTIL_MAX                 utilisation 0.609        0.542 0.533 0.551                     NaN                    NaN                       NaN
UTIL_MEAN                utilisation 0.629        0.541 0.532 0.550                     NaN                    NaN                       NaN
SHARE_OVER_LIMIT         utilisation 0.621        0.540 0.532 0.549                     NaN                    NaN                       NaN
UTIL_MEAN_12

### 8.5 The rules applied

As in notebook 07, a rare-event ratio that cannot be computed group by group is replaced by the pooled figure, observed over expected defaults.

In [12]:
decision = pd.DataFrame(index=FEATURES)
decision['family'] = [FAMILY_OF[c] for c in FEATURES]
for column, source in {'psi': shift_table['psi'], 'shift_auc': shift_table['shift_auc'], 'test_to_train_rate': shift_table['test_to_train_rate'], 'alone': new['alone'],
                       'beyond_model': new['beyond_model'], 'beyond_low': new['low'], 'beyond_high': new['high'], 'risk_ratio_beyond_model': new['risk_ratio_beyond_model'],
                       'observed_over_expected': new['observed_over_expected']}.items():
    decision[column] = source.astype(float)
decision['closest_saved_feature'] = closest['closest saved feature']
decision['correlation_with_it'] = closest['rank correlation'].astype(float)
rule1 = decision.index.isin(SHIFT_FLAGGED)
rule2_candidates = decision.index.isin(REDUNDANT_DROPPED)
rule2_saved = decision.index.isin(list(CROSS_REDUNDANT))
ratio = decision['risk_ratio_beyond_model'].fillna(decision['observed_over_expected'])
ratio_flat = ratio.between(0.9, 1.1) | ratio.isna()
includes_half = (decision['beyond_low'] <= 0.50) & (decision['beyond_high'] >= 0.50)
rule3 = (decision['alone'] < 0.51) & includes_half & ratio_flat
print('flagged by rule 1:', decision.index[rule1].tolist())
print('rule 2, among candidates:', decision.index[rule2_candidates].tolist(), '| repeating a saved feature:', {c: CROSS_REDUNDANT[c] for c in decision.index[rule2_saved]})
print('rule 3:', decision.index[rule3 & ~rule2_candidates & ~rule2_saved].tolist())
print()
print(decision.sort_values(['family', 'beyond_model'], ascending=[True, False]).to_string(float_format=lambda v: f'{v:.3f}'))

flagged by rule 1: ['MONTHS_BACK', 'LIMIT_LATEST', 'N_DEMAND']
rule 2, among candidates: ['UTIL_MEAN', 'UTIL_LATEST', 'UTIL_MEAN_12M', 'ATM_TO_LIMIT_MEAN', 'SHARE_ARREARS'] | repeating a saved feature: {}
rule 3: ['N_CARDS']

                              family   psi  shift_auc  test_to_train_rate  alone  beyond_model  beyond_low  beyond_high  risk_ratio_beyond_model  observed_over_expected         closest_saved_feature  correlation_with_it
N_DEMAND                     arrears 0.000      0.500               2.435  0.500         0.500       0.499        0.501                      NaN                   1.365           INST_N_UNPAID_OLDER                0.088
SHARE_ARREARS                arrears 0.013      0.520               0.769  0.504         0.498       0.491        0.505                    1.035                   1.013       INST_CARD_SHARE_LATE_5D                0.549
MAX_DPD                      arrears 0.012      0.520               0.769  0.523         0.497       0.491        

### The judgement on the flagged features

Three features are flagged by rule 1, and a model given all 22 tells test applicants from train applicants at 0.78, far above the 0.63 to 0.64 of the previous two tables. So the cause matters more than usual.

It is visible in when cards started. The most common start of a card is 7 to 11 months before the application in train and 11 to 15 months in test. Cards were issued in campaigns, and the test applications came later relative to them. So how far back a card's record goes (`MONTHS_BACK`) records **calendar time**: where the application fell relative to the lender's card campaigns. That is a fact about the data's timing, not the applicant, and it is the kind of difference rule 1 says to drop for. It also adds nothing beyond the model (0.495).

In [13]:
SHIFT_JUDGEMENT = {
    'MONTHS_BACK': 'drop: it records when the application fell relative to the lender\'s card campaigns (calendar time), not the applicant',
    'LIMIT_LATEST': 'keep with a warning: the limits on offer are the same in both files; the difference is in how many cards have a recent row',
    'N_DEMAND': 'keep with a warning: a card in Demand is rare (12 development applicants) and more than twice as common in test',
}
assert set(SHIFT_JUDGEMENT) == set(decision.index[rule1])
first_month = pd.read_csv(f'{RAW}/credit_card_balance.csv', usecols=['SK_ID_PREV', 'SK_ID_CURR', 'MONTHS_BALANCE']).groupby('SK_ID_PREV').agg(applicant=('SK_ID_CURR', 'first'), first=('MONTHS_BALANCE', 'min'))
first_month['file'] = np.where(first_month['applicant'].isin(TEST_IDS), 'test', 'train')
print('the five most common first months of a card (% of cards), by file:')
for label, g in first_month.groupby('file'):
    print(f"   {label}: {(g['first'].value_counts(normalize=True).head(5) * 100).round(2).to_dict()}")
del first_month

rule1_drop = decision.index.isin([c for c, j in SHIFT_JUDGEMENT.items() if j.startswith('drop')])
decision['outcome'] = np.select([rule2_candidates, rule2_saved, rule3, rule1_drop, rule1], ['dropped: redundant with another candidate', 'dropped: redundant with a saved feature', 'dropped: no signal',
                                                                                            'dropped: records calendar time', 'kept, shift warning'], 'kept')
decision.loc[decision.index.isin(FINE_DETAIL) & (decision['outcome'] == 'kept'), 'outcome'] = 'kept, fine-detail warning'
SCREENED = list(decision.index[decision['outcome'].str.startswith('kept')])
SCREENED_FEATURES = [PREFIX + c for c in SCREENED]
DROPPED_FEATURES = {PREFIX + c: o for c, o in decision['outcome'].items() if o.startswith('dropped')}
assert len(SCREENED_FEATURES) + len(DROPPED_FEATURES) == 22
print('\noutcome:')
print(decision['outcome'].value_counts().to_string())
print(f"\nkept ({len(SCREENED)}): {SCREENED}")
kept = decision.loc[SCREENED]
print(f"interval beyond the model wholly above 0.50: {int((kept['beyond_low'] > 0.50).sum())} of {len(kept)}; largest {kept['beyond_model'].max():.3f} ({kept['beyond_model'].idxmax()})")
JOINT_KEPT = joint_shift(PF.loc[both_h, SCREENED].to_numpy())
print(f"telling test from train with the kept features: {JOINT_KEPT:.4f} (all 22: {JOINT_ALL:.4f})")
print(f"what the redundancy rule kept and dropped among the utilisation features: kept {[c for c in ['UTIL_MAX', 'SHARE_OVER_LIMIT', 'SHARE_WITH_BALANCE', 'BALANCE_LATEST'] if c in SCREENED]}; "
      f"dropped UTIL_LATEST (beyond the model {decision.loc['UTIL_LATEST', 'beyond_model']:.3f}) in favour of BALANCE_LATEST ({decision.loc['BALANCE_LATEST', 'beyond_model']:.3f})")

the five most common first months of a card (% of cards), by file:
   test: {-96: 13.27, -12: 5.1, -14: 4.87, -13: 4.84, -11: 4.3}
   train: {-96: 10.85, -8: 4.88, -9: 4.44, -7: 4.17, -10: 3.89}

outcome:
outcome
kept                                         12
dropped: redundant with another candidate     5
kept, shift warning                           2
dropped: no signal                            1
dropped: records calendar time                1
kept, fine-detail warning                     1

kept (15): ['MONTHS_SINCE_LATEST_ROW', 'UTIL_MAX', 'SHARE_OVER_LIMIT', 'SHARE_WITH_BALANCE', 'BALANCE_LATEST', 'LIMIT_LATEST', 'LIMIT_GROWTH', 'SHARE_ATM', 'N_ATM_12M', 'SHARE_UNDER_MIN', 'SHARE_PAID_NOTHING', 'PAY_TO_BALANCE_MEAN', 'PAY_TO_MIN_MEAN', 'MAX_DPD', 'N_DEMAND']
interval beyond the model wholly above 0.50: 8 of 15; largest 0.544 (SHARE_ATM)


telling test from train with the kept features: 0.6575 (all 22: 0.7806)
what the redundancy rule kept and dropped among the utilisation features: kept ['UTIL_MAX', 'SHARE_OVER_LIMIT', 'SHARE_WITH_BALANCE', 'BALANCE_LATEST']; dropped UTIL_LATEST (beyond the model 0.551) in favour of BALANCE_LATEST (0.534)


### Stage 8 findings

The figures are in the cells above. In summary:

**15 of the 22 candidates go forward.** Five are near-copies of another candidate, one has no signal (the number of cards, almost always one), and one records calendar time.

**What carries over is exposure.** The kept features beyond the current model are led by cash withdrawals (0.544), the highest utilisation (0.542), the share of months over the limit (0.540) and the number of cash withdrawals in the last year (0.539). Twelve of the 22 candidates had an interval wholly above 0.50 beyond the model, against 8 of 21 for notebook 09's table and 27 of 33 for notebook 07's.

**No candidate repeats a saved feature.** The closest is the highest utilisation against the number of card instalments, at 0.75. Unlike notebook 09's table, this one is not a second view of something the model already has.

**The redundancy rule has a cost this time.** It keeps, from each group of near-copies, the feature with a value for the most applicants. Utilisation in the latest month (0.551 beyond the model, the best candidate) correlates at 0.94 with the balance in the latest month, which has a value for more card holders because it does not need a limit above zero. So the rule keeps the balance (0.534) and drops the utilisation. In notebook 07 the same kind of choice cost 0.004 of signal; here it costs 0.017. I keep to the rule: it exists so that this choice is not made by looking at scores, and the highest utilisation and the share over the limit, both kept, carry most of what the latest utilisation says. The cost is recorded for notebook 12.

**Shift is higher than for any table so far.** All 22 features together tell the files apart at 0.78. After dropping the calendar feature and the near-copies, the 15 kept features tell them apart at 0.66, close to the 0.63 and 0.64 of the previous two tables. Most of the 0.78 came from the dropped features. The test applicants hold cards more often (34% against 28%), started them later relative to the application, and use them a little less (fewer months with a balance and fewer cash withdrawals). Two features carry shift warnings, the latest limit and cards in `Demand`, and the share of months with a cash withdrawal a fine-detail warning.

**Leakage check.** Rules 1 and 2 use no outcomes. Rule 3 dropped one feature on development outcomes, the card count, which scores 0.501 alone and beyond the model. The holdout was not used.

**Expectation for notebook 12, stated now.** A gain, likely larger than notebook 10's and smaller than notebook 08's, because these features are strong but describe the 28% of applicants with a card. Because of the shift, the Kaggle result matters more than usual for this table, and a diagnostic submission without the warned features is planned.

## Stage 9: Save

In [14]:
import hashlib, subprocess, tempfile
DEFINITIONS = {
    'N_CARDS': 'Cards with monthly rows. 0 when there are none.', 'MONTHS_BACK': 'How far back the card record goes, in months.',
    'MONTHS_SINCE_LATEST_ROW': 'Months from the latest row of any card to the application.',
    'UTIL_MEAN': 'Average over months of balance over limit, where the limit is above zero, never below zero.', 'UTIL_MAX': 'The highest such utilisation.',
    'UTIL_LATEST': 'Utilisation in each card\'s latest row, if within 3 months; the highest over cards.', 'UTIL_MEAN_12M': 'Average utilisation over the last 12 months.',
    'SHARE_OVER_LIMIT': 'Share of months with a limit in which the balance exceeded it.', 'SHARE_WITH_BALANCE': 'Share of months with a balance above zero.',
    'BALANCE_LATEST': 'Balance in each card\'s latest row, if within 3 months; summed over cards.', 'LIMIT_LATEST': 'Limit in each card\'s latest row, if within 3 months; summed over cards.',
    'LIMIT_GROWTH': 'Latest limit above zero over the first limit above zero; the highest over cards.',
    'SHARE_ATM': 'Share of months with a cash withdrawal. A missing withdrawal amount counts as none.', 'ATM_TO_LIMIT_MEAN': 'Average over months with a limit of cash withdrawn over the limit.',
    'N_ATM_12M': 'Number of cash withdrawals in the last 12 months.',
    'SHARE_UNDER_MIN': 'Share of due months (a minimum above 1 recorded) in which less than the minimum minus 1 was paid.', 'SHARE_PAID_NOTHING': 'Share of due months in which nothing was paid.',
    'PAY_TO_BALANCE_MEAN': 'Average over months with a balance of the amount paid over the balance, capped at 1.', 'PAY_TO_MIN_MEAN': 'Average over due months of the amount paid over the minimum, capped at 5.',
    'SHARE_ARREARS': 'Share of months past due under the tolerance rule (SK_DPD_DEF above 0).', 'MAX_DPD': 'The largest days past due under the tolerance rule.',
    'N_DEMAND': 'Cards ever in status Demand.',
}
assert set(DEFINITIONS) == set(FEATURES)
save_train, save_test = train_table[SCREENED_FEATURES], test_table[SCREENED_FEATURES]
TRAIN_PATH, TEST_PATH, META_PATH = (Path(f'{PROCESSED}/credit_card_{x}') for x in ['train_features.parquet', 'test_features.parquet', 'features_metadata.json'])
save_train.to_parquet(TRAIN_PATH)
save_test.to_parquet(TEST_PATH)
sha = lambda path: hashlib.sha256(Path(path).read_bytes()).hexdigest()
warnings_by_feature = {PREFIX + c: f'Differs between train and test (stage 8, rule 1): {SHIFT_JUDGEMENT[c]}.' for c in SHIFT_JUDGEMENT if PREFIX + c in SCREENED_FEATURES}
warnings_by_feature.update({PREFIX + c: 'Its distribution differs between train and test in fine detail (stage 8).' for c in FINE_DETAIL if PREFIX + c in SCREENED_FEATURES and PREFIX + c not in warnings_by_feature})
metadata = {
    'description': 'Features from the monthly balances of earlier Home Credit credit cards, one row per applicant. Built in notebooks/11_credit_card_eda_features.ipynb.',
    'source': 'data/raw/credit_card_balance.csv',
    'tables': {'train': {'file': TRAIN_PATH.name, 'rows': int(len(save_train)), 'sha256': sha(TRAIN_PATH), 'row_order': 'application_train_features.parquet'},
               'test': {'file': TEST_PATH.name, 'rows': int(len(save_test)), 'sha256': sha(TEST_PATH), 'row_order': 'application_test_features.parquet'}},
    'index': 'SK_ID_CURR',
    'applicants_without_a_card': 'Every feature is missing.',
    'features': {PREFIX + c: {'family': FAMILY_OF[c], 'definition': DEFINITIONS[c], **({'warning': warnings_by_feature[PREFIX + c]} if PREFIX + c in warnings_by_feature else {})} for c in SCREENED},
    'dropped_in_screening': {c: {'reason': r.replace('dropped: ', ''), 'definition': DEFINITIONS[c.replace(PREFIX, '')]} for c, r in DROPPED_FEATURES.items()},
    'constants': {'recent_months': RECENT_MONTHS, 'latest_window_months': LATEST_WINDOW, 'pay_to_min_cap': PAY_TO_MIN_CAP, 'min_due_above': MIN_DUE_ABOVE},
    'shift_note': 'Test applicants hold cards more often (34% against 28%) and started them later relative to the application. A model given all 22 candidates told the files apart at 0.78.',
    'properties_verified': ['Each applicant\'s features depend on that applicant\'s rows only.', 'An independent rebuild of 521 applicants agrees on all 22 features.', 'No row is dated on or after the application.'],
    'selection_note': 'One feature was dropped using development outcomes (rule 3). The holdout was not used.',
    'code': 'src/features/credit_card.py',
    'versions': {'python': sys.version.split()[0], 'pandas': pd.__version__, 'numpy': np.__version__},
}
META_PATH.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
disk_train, disk_test = pd.read_parquet(TRAIN_PATH), pd.read_parquet(TEST_PATH)
checks = {
    'files equal the verified tables': disk_train.equals(save_train) and disk_test.equals(save_test),
    f'shapes 307,511 x {len(SCREENED)} and 48,744 x {len(SCREENED)}': disk_train.shape == (307511, len(SCREENED)) and disk_test.shape == (48744, len(SCREENED)),
    'row order matches the application tables': disk_train.index.equals(app_train_index) and disk_test.index.equals(app_test_index),
    'no outcome column and none of the dropped features': 'TARGET' not in disk_train.columns and not (set(DROPPED_FEATURES) & set(disk_train.columns)),
    'metadata lists the saved columns and matching checksums': list(json.loads(META_PATH.read_text(encoding='utf-8'))['features']) == list(disk_train.columns) and metadata['tables']['train']['sha256'] == sha(TRAIN_PATH),
}
MODULE = Path('../src/features/credit_card.py')
with tempfile.TemporaryDirectory() as tmp:
    for label, application_file, saved in [('train', 'application_train_features.parquet', disk_train), ('test', 'application_test_features.parquet', disk_test)]:
        out = Path(tmp) / f'{label}.parquet'
        run = subprocess.run([sys.executable, str(MODULE), f'{RAW}/credit_card_balance.csv', f'{PROCESSED}/{application_file}', str(out)], capture_output=True, text=True)
        assert run.returncode == 0, run.stderr[-2000:]
        checks[f'a separate process rebuilds the {label} file exactly ({run.stdout.strip()})'] = pd.read_parquet(out).equals(saved)
import credit_card as module
checks['the module lists the same features, dropped features and constants'] = (module.FEATURES == FEATURES and set(module.DROPPED_FEATURES) == set(DROPPED_FEATURES)
                                                                                  and (module.RECENT_MONTHS, module.LATEST_WINDOW, module.PAY_TO_MIN_CAP, module.MIN_DUE_ABOVE) == (RECENT_MONTHS, LATEST_WINDOW, PAY_TO_MIN_CAP, MIN_DUE_ABOVE))
checks['the module does not mention the target'] = 'TARGET' not in MODULE.read_text(encoding='utf-8')
for name, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {name}")
assert all(checks.values())
print('warnings carried:', list(warnings_by_feature))

PASS  files equal the verified tables
PASS  shapes 307,511 x 15 and 48,744 x 15
PASS  row order matches the application tables
PASS  no outcome column and none of the dropped features
PASS  metadata lists the saved columns and matching checksums
PASS  a separate process rebuilds the train file exactly (307511 applicants, 15 features)
PASS  a separate process rebuilds the test file exactly (48744 applicants, 15 features)
PASS  the module lists the same features, dropped features and constants
PASS  the module does not mention the target
warnings carried: ['CC_LIMIT_LATEST', 'CC_N_DEMAND', 'CC_SHARE_ATM']


### Stage 9 findings, and summary of this notebook

**Saved and verified.** Two feature tables of 15 columns, a metadata file, and `src/features/credit_card.py`, which a separate process used to rebuild both tables exactly from the raw file.

**What the table is.** 3.8 million months of 104,000 earlier credit cards, held by 28% of train applicants and 34% of test applicants. It is supplied on the same terms in train and test, and nothing in it is dated at or after the application.

**What it says.** How much of the limit an applicant is using, and whether they draw cash, are the strongest summaries any history table has produced in this project. A card over its limit at the time of the application goes with a 20% default rate. Missed minimum payments and arrears, by contrast, say nothing beyond what the model knows.

**What is new.** Unlike the monthly balances of notebook 09, almost all of it. No candidate repeats a saved feature, and the leading features keep a signal of 0.54 to 0.55 beyond the current model.

**What to watch.** The card features differ between train and test more than any table so far, partly because test applicants applied later relative to the lender's card campaigns. One feature that records that timing was dropped, and two more carry warnings.

**What I got wrong.** A consistency rule failed because the definition of a due month said less than it seemed to; the definition was tightened. And the redundancy rule dropped the best single candidate in favour of a close copy with wider coverage, at a cost I recorded rather than undid.

**Next.** Notebook 12 tests the 15 features against the model of notebook 08, with the four conditions. If they are adopted, its stage 9 should include a diagnostic submission without the warned features, because the shift is large.